# SVM vs Random Forest - CICIDS2017 (CTGAN Balanced)
Benchmark on cleaned CICIDS2017 dataset with CTGAN data augmentation and 70-30 train/test split

**Strategy:**
- Majority classes (>10,286 samples): Undersample to 50,000
- Minority classes (≤10,286 samples): CTGAN augment to 20,000

In [ ]:
!pip install ctgan -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.under_sampling import RandomUnderSampler
from ctgan import CTGAN
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Step 1: Load Data

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load cleaned CICIDS2017 dataset from Google Drive
df = pd.read_csv('/content/drive/MyDrive/data_processed/cicids2017_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]}...")
print(f"\nOriginal Class Distribution:")
print(df['Label'].value_counts())

## Step 2: Prepare Train/Test Split (70-30)

In [ ]:
# Separate features and labels
X = df.drop('Label', axis=1)
y = df['Label']
feature_columns = X.columns.tolist()

print(f"Total features: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")

# 70-30 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution (BEFORE resampling):")
print(y_train.value_counts())

## Step 3: Undersample Majority Classes to 50,000

In [ ]:
threshold = 10286
target_minority = 20000

# Undersample classes above threshold to 50,000
under_strategy = {}
for label, count in Counter(y_train).items():
    if count > threshold:
        under_strategy[label] = 50000

print(f"Undersampling classes: {list(under_strategy.keys())} -> 50,000 each")

under_sampler = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
X_train_under, y_train_under = under_sampler.fit_resample(X_train, y_train)

print(f"\nAfter undersampling: {len(y_train_under):,} samples")
print(f"\nClass distribution after undersampling:")
print(pd.Series(y_train_under).value_counts())

## Step 4: CTGAN Augmentation for Minority Classes (→ 20,000 each)

In [ ]:
# Identify minority classes (those with <= threshold samples)
class_counts = Counter(y_train_under)
minority_classes = {label: count for label, count in class_counts.items() if count <= threshold}

print("=" * 60)
print("CTGAN AUGMENTATION")
print("=" * 60)
print(f"\nMinority classes to augment to {target_minority}:")
for label, count in sorted(minority_classes.items()):
    print(f"  Label {label}: {count} -> {target_minority} (generating {target_minority - count} synthetic samples)")

# Combine X and y for CTGAN (it needs a full dataframe)
train_df = pd.DataFrame(X_train_under, columns=feature_columns)
train_df['Label'] = y_train_under.values

# Store all synthetic data
synthetic_dfs = []

for label, current_count in sorted(minority_classes.items()):
    samples_needed = target_minority - current_count
    if samples_needed <= 0:
        print(f"\nLabel {label}: Already has {current_count} samples, skipping.")
        continue
    
    print(f"\n{'='*40}")
    print(f"Training CTGAN for Label {label} ({current_count} samples -> {target_minority})")
    print(f"Generating {samples_needed} synthetic samples...")
    print(f"{'='*40}")
    
    # Extract data for this class
    class_data = train_df[train_df['Label'] == label].drop('Label', axis=1)
    
    # Train CTGAN on this class
    ctgan = CTGAN(
        epochs=300,
        batch_size=min(500, current_count),
        verbose=True
    )
    ctgan.fit(class_data)
    
    # Generate synthetic samples
    synthetic_data = ctgan.sample(samples_needed)
    synthetic_data['Label'] = label
    synthetic_dfs.append(synthetic_data)
    
    print(f"Label {label}: Generated {len(synthetic_data)} synthetic samples ✓")

print(f"\n{'='*60}")
print("CTGAN augmentation complete!")
print(f"{'='*60}")

In [ ]:
# Combine original undersampled data with synthetic data
if synthetic_dfs:
    all_synthetic = pd.concat(synthetic_dfs, ignore_index=True)
    augmented_df = pd.concat([train_df, all_synthetic], ignore_index=True)
else:
    augmented_df = train_df.copy()

# Separate back into X and y
X_train_resampled = augmented_df.drop('Label', axis=1).values
y_train_resampled = augmented_df['Label'].values

print(f"\nFinal Training set class distribution (AFTER CTGAN):")
print(pd.Series(y_train_resampled).value_counts())
print(f"\nTotal training samples: {len(y_train_resampled):,}")

## Step 5: Scale Features

In [ ]:
# Standardize features (fit on augmented training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled successfully!")
print(f"Train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Test mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

## Step 6: Train SVM (SGDClassifier with Hinge Loss)

In [ ]:
print("Training SVM (SGDClassifier with Hinge Loss)...")
svm_model = SGDClassifier(
    loss='hinge',
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    tol=1e-3,
    n_jobs=-1
)
svm_model.fit(X_train_scaled, y_train_resampled)

# Predictions
y_train_pred_svm = svm_model.predict(X_train_scaled)
y_test_pred_svm = svm_model.predict(X_test_scaled)

# Metrics
svm_metrics = {
    'Algorithm': 'SVM (SGDClassifier - Hinge Loss)',
    'Train Accuracy': accuracy_score(y_train_resampled, y_train_pred_svm),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_svm),
    'Precision': precision_score(y_test, y_test_pred_svm, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_test_pred_svm, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_test_pred_svm, average='weighted', zero_division=0)
}

print(f"SVM Training Complete!")
print(f"  Train Accuracy: {svm_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {svm_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {svm_metrics['Precision']:.4f}")
print(f"  Recall: {svm_metrics['Recall']:.4f}")
print(f"  F1-Score: {svm_metrics['F1-Score']:.4f}")

## Step 7: Train Random Forest

In [ ]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced_subsample')
rf_model.fit(X_train_resampled, y_train_resampled)

# Predictions
y_train_pred_rf = rf_model.predict(X_train_resampled)
y_test_pred_rf = rf_model.predict(X_test)

# Metrics
rf_metrics = {
    'Algorithm': 'Random Forest',
    'Train Accuracy': accuracy_score(y_train_resampled, y_train_pred_rf),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_rf),
    'Precision': precision_score(y_test, y_test_pred_rf, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_test_pred_rf, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_test_pred_rf, average='weighted', zero_division=0)
}

print(f"Random Forest Training Complete!")
print(f"  Train Accuracy: {rf_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {rf_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {rf_metrics['Precision']:.4f}")
print(f"  Recall: {rf_metrics['Recall']:.4f}")
print(f"  F1-Score: {rf_metrics['F1-Score']:.4f}")

## Step 8: Compare Results (Table)

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame([svm_metrics, rf_metrics])
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON (CTGAN Balanced Dataset)")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

## Step 9: Visualize Metrics (Line Plot)

In [ ]:
# Prepare data for visualization
metrics_data = comparison_df.set_index('Algorithm').T

# Plot 1: Line plot
fig, ax = plt.subplots(figsize=(12, 6))
metrics_data.plot(ax=ax, marker='o', linewidth=2, markersize=8)
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Metrics', fontsize=12)
ax.set_title('SVM vs Random Forest - Performance Metrics (CTGAN Balanced)', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

# Add value labels on points
for col in metrics_data.columns:
    for idx, val in enumerate(metrics_data[col]):
        ax.text(idx, val + 0.02, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Step 10: Visualize Metrics (Bar Plot)

In [ ]:
# Plot 2: Bar plot
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_to_plot = ['Train Accuracy', 'Test Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#1f77b4', '#ff7f0e']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    values = [svm_metrics[metric], rf_metrics[metric]]
    bars = ax.bar(['SVM', 'Random Forest'], values, color=colors, alpha=0.8, edgecolor='black')
    ax.set_ylabel(metric, fontsize=11)
    ax.set_ylim([0, 1.05])
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Remove extra subplot
fig.delaxes(axes[5])

fig.suptitle('SVM vs Random Forest - Detailed Metrics (CTGAN Balanced)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Step 11: Confusion Matrices

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SVM
cm_svm = confusion_matrix(y_test, y_test_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('SVM Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Random Forest
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=True)
axes[1].set_title('Random Forest Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Step 12: Classification Reports

In [ ]:
print("\n" + "="*80)
print("SVM - CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred_svm, zero_division=0))

print("\n" + "="*80)
print("RANDOM FOREST - CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred_rf, zero_division=0))

## Step 13: Summary

In [ ]:
print("\n" + "#"*80)
print("# SUMMARY")
print("#"*80)

best_f1 = 'SVM' if svm_metrics['F1-Score'] > rf_metrics['F1-Score'] else 'Random Forest'
best_acc = 'SVM' if svm_metrics['Test Accuracy'] > rf_metrics['Test Accuracy'] else 'Random Forest'

print(f"\nDataset: CICIDS2017 (Cleaned + CTGAN Balanced)")
print(f"Total Samples (Original): {len(df):,}")
print(f"Total Training Samples (After CTGAN): {len(y_train_resampled):,}")
print(f"Total Features: {X.shape[1]}")
print(f"Train/Test Split: 70/30")
print(f"Test Samples: {len(X_test):,}")
print(f"\nResampling Strategy:")
print(f"  - Classes > {threshold} samples: Undersampled to 50,000")
print(f"  - Classes <= {threshold} samples: CTGAN augmented to {target_minority}")

print(f"\n" + "-"*80)
print(f"BEST MODEL (by Test Accuracy): {best_acc}")
print(f"BEST MODEL (by F1-Score): {best_f1}")
print(f"-"*80)

print(f"\nSVM Performance:")
for key, value in svm_metrics.items():
    if key != 'Algorithm':
        print(f"  {key}: {value:.4f}")

print(f"\nRandom Forest Performance:")
for key, value in rf_metrics.items():
    if key != 'Algorithm':
        print(f"  {key}: {value:.4f}")

print(f"\n" + "#"*80)

## Step 14: Save Models to Drive

In [ ]:
import joblib
import os

save_path = '/content/drive/MyDrive/IDS_Project/saved_models_ctgan'
os.makedirs(save_path, exist_ok=True)

joblib.dump(svm_model, f'{save_path}/svm_model_ctgan.pkl')
joblib.dump(rf_model, f'{save_path}/rf_model_ctgan.pkl')
joblib.dump(scaler, f'{save_path}/scaler_ctgan.pkl')

print("All models saved to Google Drive!")
print(f"Location: {save_path}")